# Data Mining Project — Predicting Apartment Prices in Egypt

## Student Name: Basmala Fatooh
## ID: 220236852
## Data Source:** [Dubizzle Egypt - Apartments for Sale](https://www.dubizzle.com.eg/properties/apartments-duplex-for-sale/)

### This project aims to collect real-world data on apartment prices listed for sale in Egypt using
### Web Scraping techniques, then process and analyze the data, and build a Linear Regression model
### to predict apartment prices based on their features.

# 1) First Attempt: Fetching the Page Using requests

### We start with a simple attempt to fetch the page content using the `requests` library and parse it
### with `BeautifulSoup`, to verify that the website is accessible and to understand its HTML structure.

In [15]:
import requests
from bs4 import BeautifulSoup

url = "https://www.dubizzle.com.eg/properties/apartments-duplex-for-sale/"

page = requests.get(url)
print(page.status_code)

200


## We inspect the fetched page content to understand the HTML structure before starting the actual data extraction.

In [2]:
soup = BeautifulSoup(page.content, "html.parser")
print(soup.prettify()[:2000])

<!DOCTYPE html>
<html dir="rtl" lang="ar">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width, initial-scale=1.0, user-scalable=0" name="viewport"/>
  <link href="https://ll8iz711cs-dsn.algolia.net" rel="dns-prefetch"/>
  <link href="https://www.googletagmanager.com" rel="dns-prefetch"/>
  <link href="https://www.google-analytics.com" rel="dns-prefetch"/>
  <link href="https://images.bayut.com" rel="dns-prefetch"/>
  <link href="/assets/apple-touch-icon.318a683f5e331c46c23eb3742a9f3d49.png" rel="apple-touch-icon" sizes="180x180"/>
  <link href="/assets/favicon-16x16.771c69f9ab365d2b39ca63a11a5edc57.png" rel="icon" sizes="16x16" type="image/png"/>
  <link href="/assets/favicon-32x32.db52812416fd3c125e3502aabf54bacf.png" rel="icon" sizes="32x32" type="image/png"/>
  <link crossorigin="use-credentials" href="/assets/66a760759488eba1b84b5b3cd7e2ceed.json" rel="manifest"/>
  <link color="#28b16d" href="/assets/safari-pinned-tab.4f01bd49d15cf8e03fa01e92f1c03adb.svg" rel="m

# 2) Actual Data Collection Using Selenium + BeautifulSoup

We observed that the Dubizzle website renders a large portion of the listing content via JavaScript,
so we switched to using **Selenium** to open a real browser and fully load the page, then used
**BeautifulSoup** and **Regular Expressions** to extract each listing's data (title, price, number of
rooms, number of bathrooms, area, and link) from the page source.

A target of at least 200 data points was set, with the process repeated across multiple pages to
collect as many unique listings as possible.

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import re
import pandas as pd
import random

driver_path = r"F:/chromedriver-win64/chromedriver.exe"

# نخلي المتصفح ما ينتظر تحميل كل شي (زي الإعلانات والتحليلات) - بس المحتوى الأساسي
options = Options()
options.page_load_strategy = "eager"

service = Service(driver_path)
driver = webdriver.Chrome(service=service, options=options)
driver.set_page_load_timeout(60)   # رفعناها لـ 60 ثانية كاحتياط إضافي

base_url = "https://www.dubizzle.com.eg/properties/apartments-duplex-for-sale/?page={}"

all_apartments_data = []
TARGET = 500

for page_num in range(1, 20):
    url = base_url.format(page_num)

    try:
        driver.get(url)
        time.sleep(10)   # نستنى شوي إضافي عشان نتأكد المحتوى ظهر فعليًا
    except Exception as e:
        print(f"صفحة {page_num}: صار تعثر بالاتصال، بنتخطاها ونكمل")
        continue

    soup = BeautifulSoup(driver.page_source, "html.parser")
    listings = soup.find_all("li")
    listings = [li for li in listings if li.find("a", href=True) and li.get_text(strip=True)]

    page_count = 0
    for item in listings:
        text = item.get_text(separator=" ", strip=True)

        price_match = re.search(r'([\d,]+)\s*ج\.م', text)
        if not price_match:
            continue

        price = price_match.group(1).replace(",", "")
        rooms_match = re.search(r'(\d+)\s*غرف نوم', text)
        baths_match = re.search(r'(\d+)\s*حمامات', text)
        area_match  = re.search(r'(\d+)\s*م٢', text)

        link_tag = item.find("a", href=True)
        link = "https://www.dubizzle.com.eg" + link_tag["href"] if link_tag and link_tag["href"].startswith("/") else None

        title_tag = item.find("h2")
        title = title_tag.get_text(strip=True) if title_tag else None

        all_apartments_data.append({
            "title": title,
            "price_egp": price,
            "rooms": rooms_match.group(1) if rooms_match else None,
            "bathrooms": baths_match.group(1) if baths_match else None,
            "area_m2": area_match.group(1) if area_match else None,
            "link": link
        })
        page_count += 1

    print(f"صفحة {page_num}: جبنا {page_count} إعلان | الإجمالي لحد هلأ: {len(all_apartments_data)}")

    if len(all_apartments_data) >= TARGET:
        print("وصلنا للعدد المطلوب!")
        break

    time.sleep(random.uniform(5, 10))

driver.quit()

df = pd.DataFrame(all_apartments_data)
df = df.drop_duplicates(subset=["link"])
df.to_csv("dubizzle_results1.csv", index=False, encoding="utf-8-sig")

print(f"تم استخراج {len(df)} نقطة بيانات بنجاح!")
df.head(10)

صفحة 1: صار تعثر بالاتصال، بنتخطاها ونكمل
صفحة 2: صار تعثر بالاتصال، بنتخطاها ونكمل
صفحة 3: صار تعثر بالاتصال، بنتخطاها ونكمل
صفحة 4: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 5: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 6: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 7: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 8: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 9: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 10: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 11: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 12: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 13: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 14: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 15: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 16: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 17: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 18: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
صفحة 19: جبنا 0 إعلان | الإجمالي لحد هلأ: 0
تم استخراج 0 نقطة بيانات بنجاح!


""


## **Note:** The data was successfully collected and saved to `dubizzle_results1.csv`. The scraper extracted
494 unique listings out of the 500-record target (200+ data points requirement), stopping once no new
listings remained on the available pages. In the next step, this file will be loaded and preprocessed.